# 🎗️ Breast Cancer Classification using Artificial Neural Network (ANN)

**Dataset:** Wisconsin Diagnostic Breast Cancer (WDBC)  
**Task:** Binary Classification — Benign vs Malignant  
**Framework:** TensorFlow / Keras  

---
## Table of Contents
1. [Library Imports & Configuration](#1)
2. [Stage 1 — Dataset Loading & Exploration](#2)
3. [Stage 2 — Data Preprocessing](#3)
4. [Stage 3 — ANN Architecture](#4)
5. [Stage 4 — Model Training](#5)
6. [Stage 5 — Model Evaluation](#6)
7. [Stage 6 — Visualizations](#7)
8. [Stage 7 — Sample Predictions](#8)
---

## 1. Library Imports & Configuration <a id='1'></a>

In [ ]:
# ── Standard library ────────────────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings('ignore')

# ── Numerical / DataFrame ────────────────────────────────────────────────────
import numpy  as np
import pandas as pd

# ── Visualisation ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

# ── Scikit-learn ─────────────────────────────────────────────────────────────
from sklearn.datasets         import load_breast_cancer
from sklearn.model_selection  import train_test_split
from sklearn.preprocessing    import StandardScaler
from sklearn.metrics          import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    roc_curve, roc_auc_score
)
import joblib

# ── TensorFlow / Keras ───────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras.models    import Sequential
from tensorflow.keras.layers    import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow version : {tf.__version__}')
print(f'NumPy      version : {np.__version__}')
print(f'Pandas     version : {pd.__version__}')
print('✔  All libraries loaded successfully.')

## 2. Stage 1 — Dataset Loading & Exploration <a id='2'></a>

In [ ]:
# ── Load the Wisconsin Diagnostic Breast Cancer dataset ──────────────────────
raw = load_breast_cancer()

# Build a human-readable DataFrame
# sklearn encodes: 0 = malignant, 1 = benign
# We remap: 1 = Malignant, 0 = Benign  (positive class = cancer)
df = pd.DataFrame(raw.data, columns=raw.feature_names)
df['target']    = np.where(raw.target == 0, 1, 0)   # 1=Malignant, 0=Benign
df['diagnosis'] = df['target'].map({1: 'Malignant', 0: 'Benign'})

print('Dataset loaded from: sklearn.datasets.load_breast_cancer()')
print(f'Dataset shape : {df.shape}  →  ({df.shape[0]} samples × {df.shape[1]} columns)')
print()
df.head(10)

In [ ]:
# ── Feature names ────────────────────────────────────────────────────────────
print(f'Total feature columns : {len(raw.feature_names)}')
print()
for i, name in enumerate(raw.feature_names, 1):
    print(f'  {i:2d}. {name}')

In [ ]:
# ── Descriptive statistics ───────────────────────────────────────────────────
print('Descriptive Statistics (numerical features):')
df[list(raw.feature_names)].describe().T.style.background_gradient(cmap='Blues', subset=['mean','std'])

In [ ]:
# ── Missing values & duplicates ──────────────────────────────────────────────
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else '  → None found!')
print()
dups = df.duplicated().sum()
print(f'Duplicate rows : {dups}')

In [ ]:
# ── Class distribution ───────────────────────────────────────────────────────
counts = df['diagnosis'].value_counts()
print('Class distribution:')
print(counts.to_string())
print(f'\nBenign    : {counts["Benign"]}  ({counts["Benign"]/len(df)*100:.1f}%)')
print(f'Malignant : {counts["Malignant"]}  ({counts["Malignant"]/len(df)*100:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Breast Cancer Dataset — Class Distribution', fontsize=14, fontweight='bold')

counts.plot(kind='bar', ax=axes[0], color=['steelblue','tomato'], edgecolor='black', width=0.5)
axes[0].set_title('Class Count')
axes[0].set_xlabel('Diagnosis')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(counts.index, rotation=0)
for bar in axes[0].patches:
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
                 int(bar.get_height()), ha='center', va='bottom', fontweight='bold')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['steelblue','tomato'], startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Class Proportion')
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature correlation heatmap (top 15 features for readability) ────────────
top_features = list(raw.feature_names[:15])
corr = df[top_features].corr()

plt.figure(figsize=(14, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap (Mean Features)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Box plots — Mean features by diagnosis ───────────────────────────────────
mean_features = [c for c in raw.feature_names if 'mean' in c]
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
fig.suptitle('Feature Distribution by Diagnosis (Mean Features)', fontsize=13, fontweight='bold')
for ax, feat in zip(axes.flatten(), mean_features):
    sns.boxplot(data=df, x='diagnosis', y=feat, palette={'Benign':'steelblue','Malignant':'tomato'}, ax=ax)
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel('')
plt.tight_layout()
plt.show()

## 3. Stage 2 — Data Preprocessing <a id='3'></a>

In [ ]:
# ── Separate features (X) and target (y) ────────────────────────────────────
FEATURES = list(raw.feature_names)
X = df[FEATURES].values   # shape: (569, 30)
y = df['target'].values   # shape: (569,)   1=Malignant, 0=Benign

print(f'Feature matrix shape : {X.shape}')
print(f'Target vector shape  : {y.shape}')
print(f'Target class balance : Benign={np.sum(y==0)}, Malignant={np.sum(y==1)}')

In [ ]:
# ── Stratified 80/20 train-test split ───────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
print(f'Training set : {X_train.shape[0]} samples')
print(f'Test set     : {X_test.shape[0]}  samples')
print(f'Train class  → Benign: {np.sum(y_train==0)}, Malignant: {np.sum(y_train==1)}')
print(f'Test class   → Benign: {np.sum(y_test==0)},  Malignant: {np.sum(y_test==1)}')

In [ ]:
# ── StandardScaler: fit on TRAIN only → prevent data leakage ────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform
X_test_scaled  = scaler.transform(X_test)        # transform only

# Save the scaler
os.makedirs('../models', exist_ok=True)
joblib.dump(scaler, '../models/scaler.pkl')
print('✔  Scaler fitted and saved to models/scaler.pkl')
print(f'   Feature mean (first 3) : {scaler.mean_[:3].round(4)}')
print(f'   Feature std  (first 3) : {scaler.scale_[:3].round(4)}')

In [ ]:
# ── Validation split from training data (15% of train) ──────────────────────
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_scaled, y_train,
    test_size=0.15, stratify=y_train, random_state=SEED
)
print(f'Train   : {X_tr.shape[0]} samples')
print(f'Val     : {X_val.shape[0]} samples')
print(f'Test    : {X_test_scaled.shape[0]} samples')

## 4. Stage 3 — ANN Model Architecture <a id='4'></a>

In [ ]:
def build_ann(input_dim: int = 30) -> Sequential:
    """
    Build the Artificial Neural Network.

    Architecture
    ────────────
    Input  (30 features)
      ↓
    Dense(64, ReLU)
      ↓
    Dropout(0.3)
      ↓
    Dense(32, ReLU)
      ↓
    Dropout(0.2)
      ↓
    Dense(1, Sigmoid)  ← Binary output probability
    """
    model = Sequential([
        # Hidden layer 1 — 64 neurons, ReLU activation
        Dense(64, activation='relu', input_shape=(input_dim,), name='dense_1'),
        Dropout(0.3, name='dropout_1'),       # regularise: randomly drop 30% of neurons

        # Hidden layer 2 — 32 neurons, ReLU activation
        Dense(32, activation='relu', name='dense_2'),
        Dropout(0.2, name='dropout_2'),       # regularise: randomly drop 20% of neurons

        # Output layer — 1 neuron, Sigmoid → probability in [0, 1]
        Dense(1, activation='sigmoid', name='output'),
    ], name='BreastCancer_ANN')

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
        ]
    )
    return model

model = build_ann(input_dim=X_tr.shape[1])
model.summary()

## 5. Stage 4 — Model Training <a id='5'></a>

In [ ]:
# ── Callbacks ────────────────────────────────────────────────────────────────
MODEL_PATH = '../models/breast_cancer_ann.keras'

callbacks = [
    # Stop training when val_loss stops improving for 12 consecutive epochs
    EarlyStopping(
        monitor='val_loss',
        patience=12,
        restore_best_weights=True,  # restore to best checkpoint weights
        verbose=1
    ),
    # Save the best model weights during training
    ModelCheckpoint(
        filepath=MODEL_PATH,
        monitor='val_loss',
        save_best_only=True,
        verbose=0
    )
]

print('Starting ANN Training...')
print('=' * 55)

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=16,
    callbacks=callbacks,
    verbose=1
)

print(f'\n✔  Training complete. Model saved to: {MODEL_PATH}')
print(f'   Total epochs run: {len(history.history["loss"])}')

## 6. Stage 5 — Model Evaluation <a id='6'></a>

In [ ]:
# ── Load best saved model ────────────────────────────────────────────────────
best_model = tf.keras.models.load_model(MODEL_PATH)

# ── Predict on unseen test set ───────────────────────────────────────────────
y_pred_prob  = best_model.predict(X_test_scaled).flatten()
y_pred_class = (y_pred_prob >= 0.5).astype(int)

# ── Calculate metrics ────────────────────────────────────────────────────────
acc  = accuracy_score(y_test, y_pred_class)
prec = precision_score(y_test, y_pred_class)
rec  = recall_score(y_test, y_pred_class)
f1   = f1_score(y_test, y_pred_class)
auc  = roc_auc_score(y_test, y_pred_prob)
cm   = confusion_matrix(y_test, y_pred_class)

print('=' * 55)
print('  FINAL MODEL EVALUATION (Test Set)')
print('=' * 55)
print(f'  Accuracy   : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision  : {prec:.4f}')
print(f'  Recall     : {rec:.4f}  ← Malignant sensitivity')
print(f'  F1-score   : {f1:.4f}')
print(f'  AUC Score  : {auc:.4f}')
print()
print('  Classification Report:')
print(classification_report(y_test, y_pred_class, target_names=['Benign', 'Malignant']))

tn, fp, fn, tp = cm.ravel()
print(f'  Confusion Matrix values:')
print(f'    True  Positives (Malignant correctly identified) : {tp}')
print(f'    False Negatives (Malignant missed)               : {fn}  ← Critical!')
print(f'    True  Negatives (Benign correctly identified)    : {tn}')
print(f'    False Positives (Benign misclassified)           : {fp}')

## 7. Stage 6 — Visualizations <a id='7'></a>

In [ ]:
# ── Training vs Validation Accuracy ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ANN Training History', fontsize=14, fontweight='bold')

axes[0].plot(history.history['accuracy'],     label='Train',      lw=2, color='royalblue')
axes[0].plot(history.history['val_accuracy'], label='Validation', lw=2, color='darkorange', linestyle='--')
axes[0].set_title('Accuracy over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train',      lw=2, color='royalblue')
axes[1].plot(history.history['val_loss'], label='Validation', lw=2, color='darkorange', linestyle='--')
axes[1].set_title('Loss over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Binary Cross-Entropy Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion Matrix Heatmap ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign', 'Malignant'],
            yticklabels=['Benign', 'Malignant'],
            ax=ax, linewidths=0.5, annot_kws={'size': 16, 'weight': 'bold'})
ax.set_title('Confusion Matrix — ANN Test Results', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('Actual Label',    fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── ROC Curve ────────────────────────────────────────────────────────────────
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, lw=2.5, color='darkorange',
        label=f'ANN (AUC = {auc:.4f})')
ax.plot([0, 1], [0, 1], lw=1.5, color='navy', linestyle='--',
        label='Random Classifier (AUC = 0.50)')
ax.fill_between(fpr, tpr, alpha=0.1, color='darkorange')
ax.set_title('ROC Curve — Breast Cancer ANN', fontsize=13, fontweight='bold')
ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11)
ax.set_ylabel('True Positive Rate (Sensitivity / Recall)', fontsize=11)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Stage 7 — Sample Predictions <a id='8'></a>

In [ ]:
# ── Display 15 sample predictions from the test set ─────────────────────────
label_map = {0: 'Benign', 1: 'Malignant'}

results = []
for i in range(min(15, len(y_test))):
    results.append({
        'Sample #':          i + 1,
        'Actual Class':      label_map[y_test[i]],
        'Predicted Class':   label_map[y_pred_class[i]],
        'Prob (Malignant)':  round(float(y_pred_prob[i]), 4),
        'Status':            '✓ Correct' if y_test[i] == y_pred_class[i] else '✗ Wrong'
    })

results_df = pd.DataFrame(results)

def colour_status(val):
    return 'color: green; font-weight: bold' if '✓' in str(val) else 'color: red; font-weight: bold'

results_df.style.applymap(colour_status, subset=['Status'])

In [ ]:
# ── Summary card ─────────────────────────────────────────────────────────────
print('=' * 55)
print('  PROJECT SUMMARY')
print('=' * 55)
print(f'  Dataset        : Wisconsin Diagnostic Breast Cancer')
print(f'  Total Samples  : {len(df)}')
print(f'  Features       : 30 numerical')
print(f'  Model          : ANN (Dense 64→32→1)')
print(f'  Optimizer      : Adam')
print(f'  Loss Function  : Binary Cross-Entropy')
print(f'  Epochs Run     : {len(history.history["loss"])} (with EarlyStopping)')
print()
print(f'  ── Test Results ──')
print(f'  Accuracy       : {acc*100:.2f}%')
print(f'  Precision      : {prec:.4f}')
print(f'  Recall         : {rec:.4f}  (Malignant sensitivity)')
print(f'  F1-score       : {f1:.4f}')
print(f'  AUC Score      : {auc:.4f}')
print(f'  False Negatives: {fn}  (malignant cases missed)')
print('=' * 55)